# Server Test Code

In [1]:
import os
import json
import logging
from typing import List, Dict, Optional
from firecrawl import FirecrawlApp
from urllib.parse import urlparse
from datetime import datetime
from mcp.server.fastmcp import FastMCP

from dotenv import load_dotenv

In [2]:
load_dotenv()
SCRAPE_DIR = "scraped_content"
fc_api_key = os.getenv('FIRECRAWL_API_KEY')
app = FirecrawlApp(api_key=fc_api_key)

In [3]:
path = os.path.join(SCRAPE_DIR)
os.makedirs(path, exist_ok=True)
metadata_file = os.path.join(path, "scraped_metadata.json")

In [4]:
# Load existing metadata
try:
    with open(metadata_file, 'r') as file:
        scraped_metadata = json.load(file)
except (FileNotFoundError, json.JSONDecodeError):
    scraped_metadata = {}                

# Create placeholder for successful scrapes
successful_scrape = []

In [7]:
formats = ['markdown', 'html']
provider_name = "Yahoo"
url = "www.yahoo.com"

# Call Firecrawl
scrape_result = app.scrape(url, formats=formats).model_dump()

# Update metadata
metadata = {
    "provider": provider_name,
    "url": url,
    "domain": urlparse(url).netloc,
    "scraped_at": datetime.now().isoformat(),
}

In [8]:
scrape_result

{'markdown': "Advertisement\n\nAdvertisement\n\nAdvertisement\n\n1. ![](https://s.yimg.com/lo/mysterio/api/26711E1D6320AED553A1B4466F3BEC5E2036121834FA4FF6DCA8027B0E8CE4BF/subgraphmysterio/smartcrop_w124_h124_faces;quality_90/https:%2F%2Fmedia.zenfs.com%2Fen%2Fcnn_articles_875%2F46beac062744c8aec15a03ac68b2705f)\n\n1\n\n\n\n[David Allan Coe, outlaw country singer, dies at 86](https://www.yahoo.com/entertainment/music/articles/david-allan-coe-wrote-job-093042000.html)\n\n2. ![](https://s.yimg.com/lo/mysterio/api/34B0E29A9C55C3819DCD7CD7BC45A50FBD33157A9CD9DBF3EA715D52BE92320E/subgraphmysterio/smartcrop_w124_h124_faces;quality_90/https:%2F%2Fs.yimg.com%2Fos%2Fcreatr-uploaded-images%2F2026-04%2F25b34460-44b9-11f1-9e37-0da889483db3)\n\n2\n\n\n\n[House passes bill to fund DHS and end 76-day shutdown](https://www.yahoo.com/news/articles/congress-ends-record-shattering-dhs-171450121.html)\n\n\n\nNew\n\n3. ![](https://s.yimg.com/lo/mysterio/api/FB843DBD7B3333B68D559D40720EDC01AE9DE8815AFF0AC57

In [9]:
scrape_result.get('success', False)

False

In [10]:
scrape_result.get('markdown', '')

"Advertisement\n\nAdvertisement\n\nAdvertisement\n\n1. ![](https://s.yimg.com/lo/mysterio/api/26711E1D6320AED553A1B4466F3BEC5E2036121834FA4FF6DCA8027B0E8CE4BF/subgraphmysterio/smartcrop_w124_h124_faces;quality_90/https:%2F%2Fmedia.zenfs.com%2Fen%2Fcnn_articles_875%2F46beac062744c8aec15a03ac68b2705f)\n\n1\n\n\n\n[David Allan Coe, outlaw country singer, dies at 86](https://www.yahoo.com/entertainment/music/articles/david-allan-coe-wrote-job-093042000.html)\n\n2. ![](https://s.yimg.com/lo/mysterio/api/34B0E29A9C55C3819DCD7CD7BC45A50FBD33157A9CD9DBF3EA715D52BE92320E/subgraphmysterio/smartcrop_w124_h124_faces;quality_90/https:%2F%2Fs.yimg.com%2Fos%2Fcreatr-uploaded-images%2F2026-04%2F25b34460-44b9-11f1-9e37-0da889483db3)\n\n2\n\n\n\n[House passes bill to fund DHS and end 76-day shutdown](https://www.yahoo.com/news/articles/congress-ends-record-shattering-dhs-171450121.html)\n\n\n\nNew\n\n3. ![](https://s.yimg.com/lo/mysterio/api/FB843DBD7B3333B68D559D40720EDC01AE9DE8815AFF0AC578053E8B47770F

In [11]:
scrape_result.get('html', '')

'<!DOCTYPE html><html lang="en-US" class="uds-color-mode-light" style="color-scheme: light;"><body class="__variable_b8938d __variable_8e7273 __variable_6d623b bg-secondary font-yahoosans text-primary" id="news-content-app"><div><!--$--><!--/$--></div><div id="benji-dashboard-root"></div><div class="grid min-h-dvh grid-cols-1 grid-rows-[auto_1fr_auto] nca-homepage-en-US narrow-page relative z-1 bg-secondary"><div><div style="--feature-bar-margin:0"><div class="border-y border-y-tertiary bg-always-black/5 dark:bg-always-black/33 flex items-center overflow-x-hidden"><div class="relative z-0 items-center justify-center mx-auto my-0 ad-container" id="_R_6ckoivb8lb_" style="height:50px;width:320px"><div class="hidden" id="top_center-frontpage_0"></div><div class="flex size-full items-center justify-center text-center leading-3 bg-accent/2" style="height:50px;width:320px" qlsd97ct2="">Advertisement</div></div><div class="relative z-0 items-center justify-center mx-auto my-0 ad-container" id=

In [12]:
scrape_result['metadata'].get('title', '')

'Yahoo | Mail, Weather, Search, Politics, News, Finance, Sports & Videos'

In [13]:
scrape_result['metadata'].get('description', '')

'Latest news coverage, email, free stock quotes, live scores and video are just the beginning. Discover more every day at Yahoo!'

---

# Client Test Code

In [15]:
import asyncio
import json
import logging
import os
import shutil
from contextlib import AsyncExitStack
from typing import Any, List, Dict, TypedDict
from datetime import datetime, timedelta
from pathlib import Path
import re

from dotenv import load_dotenv
from anthropic import Anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [16]:
class ToolDefinition(TypedDict):
    name: str
    description: str
    input_schema: dict


class Configuration:
    """Manages configuration and environment variables for the MCP client."""

    def __init__(self) -> None:
        """Initialize configuration with environment variables."""
        self.load_env()
        self.api_key = os.getenv("ANTHROPIC_API_KEY")

    @staticmethod
    def load_env() -> None:
        """Load environment variables from .env file."""
        load_dotenv()

    @staticmethod
    def load_config(file_path: str | Path) -> dict[str, Any]:
        """Load server configuration from JSON file.

        Args:
            file_path: Path to the JSON configuration file.

        Returns:
            Dict containing server configuration.

        Raises:
            FileNotFoundError: If configuration file doesn't exist.
            JSONDecodeError: If configuration file is invalid JSON.
            ValueError: If configuration file is missing required fields.
        """
        try:
            with open(file_path, 'r') as f:
                try:
                    config = json.load(f)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSON in configuration file: {e}")
        except (FileNotFoundError, json.JSONDecodeError) as e:
            raise ValueError(f"Error loading configuration file: {e}")

        if "mcpServers" not in config.keys():
            raise ValueError("Configuration file must contain 'mcpServers' field")
        
        return config

    @property
    def anthropic_api_key(self) -> str:
        """Get the Anthropic API key.

        Returns:
            The API key as a string.

        Raises:
            ValueError: If the API key is not found in environment variables.
        """
        if not self.api_key:
            raise ValueError("ANTHROPIC_API_KEY not found in environment variables")
        return self.api_key


class Server:
    """Manages MCP server connections and tool execution."""

    def __init__(self, name: str, config: dict[str, Any]) -> None:
        self.name: str = name
        self.config: dict[str, Any] = config
        self.stdio_context: Any | None = None
        self.session: ClientSession | None = None
        self._cleanup_lock: asyncio.Lock = asyncio.Lock()
        self.exit_stack: AsyncExitStack = AsyncExitStack()

    async def initialize(self) -> None:
        """Initialize the server connection."""
        command = shutil.which("npx") if self.config["command"] == "npx" else self.config["command"]
        if command is None:
            raise ValueError("The command must be a valid string and cannot be None.")

        # complete params
        server_params = StdioServerParameters(
            command=command,
            args=self.config["args"],
            env={**os.environ, **self.config["env"]} if self.config.get("env") else None
        )

        try:
            stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
            read, write = stdio_transport
            session = await self.exit_stack.enter_async_context(ClientSession(read, write))
            await session.initialize()
            self.session = session
            logging.info(f"✓ Server '{self.name}' initialized")
        except Exception as e:
            logging.error(f"Error initializing server {self.name}: {e}")
            await self.cleanup()
            raise

    async def list_tools(self) -> List[ToolDefinition]:
        """List available tools from the server.

        Returns:
            A list of available tool definitions.

        Raises:
            RuntimeError: If the server is not initialized.
        """
        
        if not self.session:
            raise RuntimeError("Server is not initialized")

        tools_response = await self.session.list_tools()
        tools = []
        for tool in tools_response.tools:
            tools.append({
            'name': tool.name,
            'description': tool.description,
            'input_schema': tool.inputSchema
            })
        
        return tools

    async def execute_tool(
        self,
        tool_name: str,
        arguments: dict[str, Any],
        retries: int = 2,
        delay: float = 1.0,
    ) -> Any:
        """Execute a tool with retry mechanism.

        Args:
            tool_name: Name of the tool to execute.
            arguments: Tool arguments.
            retries: Number of retry attempts.
            delay: Delay between retries in seconds.

        Returns:
            Tool execution result.

        Raises:
            RuntimeError: If server is not initialized.
            Exception: If tool execution fails after all retries.
        """
        if not self.session:
            raise RuntimeError("Server is not initialized")

        attempt = 0
        while attempt <= retries:
            try:
                logging.info(f"Executing {tool_name}...")
                result = await self.session.call_tool(name=tool_name, arguments=arguments)
                logging.info(f"✓ Tool '{tool_name}' executed successfully on attempt {attempt + 1}")
                return result
            except Exception as e:
                logging.warning(f"Attempt {attempt + 1} failed for tool '{tool_name}': {e}")
                if attempt == retries:
                    logging.error(f"All {retries + 1} attempts failed for tool '{tool_name}'")
                    raise
                await asyncio.sleep(delay)
                attempt += 1

    async def cleanup(self) -> None:
        """Clean up server resources."""
        async with self._cleanup_lock:
            try:
                await self.exit_stack.aclose()
                self.session = None
                self.stdio_context = None
            except Exception as e:
                logging.error(f"Error during cleanup of server {self.name}: {e}")


In [17]:
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")

In [18]:
file_path = "./server_config.json"
with open(file_path, 'r') as f:
    config_all = json.load(f)

In [19]:
config_all

{'mcpServers': {'llm_inference': {'command': 'uv',
   'args': ['run', './server.py']},
  'sqlite': {'command': 'uvx',
   'args': ['mcp-server-sqlite', '--db-path', './test.db']},
  'filesystem': {'command': 'npx',
   'args': ['-y', '@modelcontextprotocol/server-filesystem', '.']}}}

In [20]:
config_all['mcpServers'].keys()

dict_keys(['llm_inference', 'sqlite', 'filesystem'])

In [21]:
config = config_all['mcpServers']['sqlite']
config

{'command': 'uvx', 'args': ['mcp-server-sqlite', '--db-path', './test.db']}

In [22]:
# Initialize Server
command = shutil.which("npx") if config["command"] == "npx" else config["command"]

server_params = StdioServerParameters(
    command=command,
    args=config["args"],
    env={**os.environ, **config["env"]} if config.get("env") else None
)

exit_stack = AsyncExitStack()

In [23]:
stdio_transport = await exit_stack.enter_async_context(stdio_client(server_params))
read, write = stdio_transport
session = await exit_stack.enter_async_context(ClientSession(read, write))
await session.initialize()

InitializeResult(meta=None, protocolVersion='2025-11-25', capabilities=ServerCapabilities(experimental={}, logging=None, prompts=PromptsCapability(listChanged=False), resources=ResourcesCapability(subscribe=False, listChanged=False), tools=ToolsCapability(listChanged=False), completions=None, tasks=None), serverInfo=Implementation(name='sqlite', title=None, version='0.1.0', websiteUrl=None, icons=None), instructions=None)

In [24]:
tools_response = await session.list_tools()

In [25]:
tools_response

ListToolsResult(meta=None, nextCursor=None, tools=[Tool(name='read_query', title=None, description='Execute a SELECT query on the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'SELECT SQL query to execute'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='write_query', title=None, description='Execute an INSERT, UPDATE, or DELETE query on the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'SQL query to execute'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='create_table', title=None, description='Create a new table in the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'CREATE TABLE SQL statement'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=N

In [26]:
tools_response.tools

[Tool(name='read_query', title=None, description='Execute a SELECT query on the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'SELECT SQL query to execute'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='write_query', title=None, description='Execute an INSERT, UPDATE, or DELETE query on the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'SQL query to execute'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='create_table', title=None, description='Create a new table in the SQLite database', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'CREATE TABLE SQL statement'}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='list_tables', 

In [27]:
tools = []
for tool in tools_response.tools:
    tools.append({
    'name': tool.name,
    'description': tool.description,
    'input_schema': tool.inputSchema
    })

In [28]:
tools

[{'name': 'read_query',
  'description': 'Execute a SELECT query on the SQLite database',
  'input_schema': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'SELECT SQL query to execute'}},
   'required': ['query']}},
 {'name': 'write_query',
  'description': 'Execute an INSERT, UPDATE, or DELETE query on the SQLite database',
  'input_schema': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'SQL query to execute'}},
   'required': ['query']}},
 {'name': 'create_table',
  'description': 'Create a new table in the SQLite database',
  'input_schema': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'CREATE TABLE SQL statement'}},
   'required': ['query']}},
 {'name': 'list_tables',
  'description': 'List all tables in the SQLite database',
  'input_schema': {'type': 'object', 'properties': {}}},
 {'name': 'describe_table',
  'description': 'Get the schema information for a specifi

In [29]:
anthropic = Anthropic(api_key=api_key)

In [31]:
config = Configuration()
config_file = "./server_config.json"
server_config = config.load_config(config_file)

servers = [Server(name, srv_config) for name, srv_config in server_config["mcpServers"].items()]

In [32]:
servers

In [33]:
servers[0].name

'llm_inference'

In [35]:
await servers[2].initialize()

In [36]:
for server in servers:
    await server.initialize()
    if "sqlite" in server.name.lower():
        sqlite_server = server

In [37]:
tool_to_server = {}
available_tools = []
for server in servers:
    tools = await server.list_tools()
    available_tools.extend(tools)
    for tool in tools:
        tool_to_server[tool["name"]] = server.name

In [65]:
tool_to_server

{'scrape_websites': 'llm_inference',
 'extract_scraped_info': 'llm_inference',
 'read_query': 'sqlite',
 'write_query': 'sqlite',
 'create_table': 'sqlite',
 'list_tables': 'sqlite',
 'describe_table': 'sqlite',
 'append_insight': 'sqlite',
 'read_file': 'filesystem',
 'read_text_file': 'filesystem',
 'read_media_file': 'filesystem',
 'read_multiple_files': 'filesystem',
 'write_file': 'filesystem',
 'edit_file': 'filesystem',
 'create_directory': 'filesystem',
 'list_directory': 'filesystem',
 'list_directory_with_sizes': 'filesystem',
 'directory_tree': 'filesystem',
 'move_file': 'filesystem',
 'search_files': 'filesystem',
 'get_file_info': 'filesystem',
 'list_allowed_directories': 'filesystem'}

In [66]:
query = "Can you scrape the website www.yahoo.com and extract the title and description of the page?"
messages = [{'role': 'user', 'content': query}]
response = anthropic.messages.create(
    max_tokens=2024,
    model='claude-sonnet-4-6', 
    tools=available_tools,
    messages=messages
)

full_response = ""
source_url = None
used_web_search = False

process_query = True

In [67]:
response

Message(id='msg_012Bz8b9B2K32TsmMDWZrk5Z', container=None, content=[TextBlock(citations=None, text='Sure! Let me start by scraping the website **www.yahoo.com** for you!', type='text'), ToolUseBlock(id='toolu_01HiLVrSCrT3yGjs9ymzAe6q', caller=DirectCaller(type='direct'), input={'websites': {'yahoo': 'https://www.yahoo.com'}, 'formats': ['markdown', 'html']}, name='scrape_websites', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=3381, output_tokens=108, server_tool_use=None, service_tier='standard'))

In [79]:
response.content[1]

ToolUseBlock(id='toolu_01HiLVrSCrT3yGjs9ymzAe6q', caller=DirectCaller(type='direct'), input={'websites': {'yahoo': 'https://www.yahoo.com'}, 'formats': ['markdown', 'html']}, name='scrape_websites', type='tool_use')

In [82]:
response.content[1].name

'scrape_websites'

In [70]:
response.type

'message'

In [68]:
response.content[0].type

'text'

In [69]:
response.content[0].text

'Sure! Let me start by scraping the website **www.yahoo.com** for you!'

In [ ]:
response

In [41]:
async def _get_structured_extraction(self, prompt: str) -> str:
    """Use Claude to extract structured data."""
    try:
        response = self.anthropic.messages.create(
            max_tokens=1024,
            model='claude-sonnet-4-5-20250929',
            messages=[{'role': 'user', 'content': prompt}]
        )
        
        text_content = ""
        for content in response.content:
            if content.type == 'text':
                text_content += content.text
        
        return text_content.strip()
                
    except Exception as e:
        logging.error(f"Error in structured extraction: {e}")
        return '{"error": "extraction failed"}'

In [40]:
extraction_prompt = f"""
Analyze this text and extract pricing information in JSON format:

Text: {llm_response}

Extract pricing plans with this structure:
{{
    "company_name": "company name",
    "plans": [
        {{
            "plan_name": "plan name",
            "input_tokens": number or null,
            "output_tokens": number or null,
            "currency": "USD",
            "billing_period": "monthly/yearly/one-time",
            "features": ["feature1", "feature2"],
            "limitations": "any limitations mentioned",
            "query": "the user's query"
        }}
    ]
}}

Return only valid JSON, no other text. Do not return your response enclosed in ```json```
"""

extraction_response = await _get_structured_extraction(extraction_prompt)

NameError: name 'llm_response' is not defined

In [39]:
extraction_response

NameError: name 'extraction_response' is not defined